# Methane dimer at 36 qubits - SQD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhargav2603/qubit_run/blob/main/studies/methane-dimer-36q/colab.ipynb)

Reproducing the 36-qubit experiment in [arXiv:2410.09209](https://arxiv.org/abs/2410.09209),
*Accurate quantum-centric simulations of supramolecular interactions* (Commun. Phys. 2025).

**CAS(16e,16o)/aug-cc-pVQZ | 165,636,900 determinants | 32 + 4 ancilla = 36 qubits**

Runs on any Linux Jupyter host with Python >= 3.11 - **Google Colab** and
**qBraid Lab** are both detected automatically. Not Windows: `pyscf` and `ffsim`
ship no Windows wheels.

**Memory is the binding constraint.** Section 10 (CASCI) needs ~15 GiB:

| host | RAM | gets you |
|---|---|---|
| Colab (free) | 12.7 GiB | everything except section 10 |
| Colab (Pro, High-RAM) | ~51 GiB | all of it |
| qBraid Free / Standard | 4 / 8 GiB | not enough; stop after section 5 |
| qBraid Pro (Large) | 25 GiB | all of it |

Sections 0-3 are the bootstrap and cost nothing. Section 4 (`validate`) is the
gate: it exercises the whole pipeline in seconds. Everything after it costs real
CPU, so read the timing note in each header before running it.

## 0 - Environment check

Run this first. It fails here, loudly, rather than obscurely three cells down.

In [ ]:
import os, platform, shutil, sys

print('python  ', sys.version.split()[0])
print('platform', platform.platform())

assert platform.system() == 'Linux', (
    'pyscf and ffsim publish no Windows wheels; this notebook needs a Linux '
    'Jupyter host (Colab and qBraid Lab both qualify).')
assert sys.version_info >= (3, 11), (
    f'ffsim 0.0.84 requires Python >= 3.11, got '
    f'{sys.version_info.major}.{sys.version_info.minor}. On qBraid, build the '
    'environment from a newer interpreter (see the qBraid note under section 1).')

if os.path.isdir('/content'):
    HOST, BASE = 'colab', '/content'
elif os.path.isdir(os.path.expanduser('~/.qbraid')):
    HOST, BASE = 'qbraid', os.path.expanduser('~')
else:
    HOST, BASE = 'other', os.path.expanduser('~')

ram_gib  = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
disk_gib = shutil.disk_usage(BASE).free / 2**30
print(f'host     {HOST}  (working under {BASE})')
print(f'RAM      {ram_gib:.1f} GiB')
print(f'CPUs     {os.cpu_count()}')
print(f'disk     {disk_gib:.1f} GiB free')

# pyscf caps itself at 4000 MB regardless of the machine, which sends the
# aug-cc-pVQZ density fitting out to disk. Must be set before pyscf is imported;
# run.py subprocesses inherit it.
os.environ['PYSCF_MAX_MEMORY'] = str(int(max(2000, ram_gib * 1024 * 0.7)))
print(f"PYSCF_MAX_MEMORY {os.environ['PYSCF_MAX_MEMORY']} MB")

if ram_gib < 15:
    print()
    print(f'NOTE: {ram_gib:.1f} GiB is not enough for section 10 (CASCI, ~15 GiB).')
    print('      Sections 0-9 and 11 still work. To get section 10:')
    print('      Colab  -> Runtime -> Change runtime type -> High-RAM')
    print('      qBraid -> Pro tier, Large session machine (8 vCPU / 25 GB)')
if ram_gib < 6:
    print()
    print(f'WARNING: {ram_gib:.1f} GiB will also struggle from section 6 on.')
    print('         The ffsim sector statevector alone is 2.47 GiB.')

## 1 - Install (pinned)

Pinned to the versions this folder was written against; `requirements.txt` says
why. A couple of minutes. `%pip` (not `!pip`) installs into *this kernel*, which
is what makes the cell correct on qBraid as well as Colab.

> **qBraid only:** top-level installs do not survive a session. Do this once in
> a Terminal instead, then pick the `methane36` kernel for this notebook:
> ```bash
> python3 -m venv ~/methane_env && source ~/methane_env/bin/activate
> pip install -q ipykernel pyscf==2.14.0 ffsim==0.0.84 'qiskit>=2.0,<3' qiskit-addon-sqd==0.13.1
> python -m ipykernel install --user --name methane36 --display-name "Python 3 [methane36]"
> ```
> `~` is persistent storage on qBraid, so that survives restarts and you skip
> this cell from then on.

In [ ]:
%pip install -q pyscf==2.14.0 ffsim==0.0.84 'qiskit>=2.0,<3' qiskit-addon-sqd==0.13.1

Verify in a *separate* cell, so the imports happen after pip has finished. If
this raises, restart the kernel (Colab: **Runtime -> Restart session**; qBraid:
**Kernel -> Restart Kernel**) and run it again - pip may have replaced a package
that was already imported.

In [ ]:
import ffsim, pyscf, qiskit, qiskit_addon_sqd

print('pyscf', pyscf.__version__, '| ffsim', ffsim.__version__,
      '| qiskit', qiskit.__version__, '| sqd', qiskit_addon_sqd.__version__)

expected = {'pyscf': '2.14.0', 'ffsim': '0.0.84', 'qiskit_addon_sqd': '0.13.1'}
actual   = {'pyscf': pyscf.__version__, 'ffsim': ffsim.__version__,
            'qiskit_addon_sqd': qiskit_addon_sqd.__version__}
drift = {k: (v, actual[k]) for k, v in expected.items() if actual[k] != v}
assert not drift, f'version drift (expected, got): {drift}'
assert qiskit.__version__.startswith('2.'), f'need qiskit 2.x, got {qiskit.__version__}'
print()
print('versions match the pins.')

## 2 - Get the code

Clones the public repo under the host's working root and drops you inside
`studies/methane-dimer-36q/`. Safe to re-run: it pulls instead of re-cloning.

In [ ]:
import os, subprocess, sys

REPO   = 'https://github.com/bhargav2603/qubit_run.git'
NAME   = 'qubit_run'
FOLDER = 'studies/methane-dimer-36q'
CHECKOUT = os.path.join(BASE, NAME)
ROOT     = os.path.join(CHECKOUT, FOLDER)

if os.path.isdir(os.path.join(CHECKOUT, '.git')):
    subprocess.run(['git', '-C', CHECKOUT, 'pull', '--ff-only'], check=False)
elif not os.path.isdir(ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, CHECKOUT], check=True)

assert os.path.isdir(ROOT), (
    f'{ROOT} not found. If the repo is private, upload the {FOLDER}/ folder into '
    'the session with the file browser and point ROOT at it.')

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('cwd =', os.getcwd())
print(sorted(f for f in os.listdir('.') if f.endswith('.py')))

### 2b - Keeping results across sessions

`data/` holds the aug-cc-pVQZ integral cache (expensive) and every result file
that `verify` and `report` read back, so losing it costs real hours.

- **qBraid:** nothing to do. `~` is persistent storage (15 / 50 / 150 GB by
  tier), so the checkout and its `data/` are already durable.
- **Colab:** the filesystem is discarded on disconnect. Set `USE_DRIVE = True`
  below to symlink `data/` into Drive before you start anything expensive.

In [ ]:
USE_DRIVE = False   # Colab only; ignored elsewhere

if HOST == 'colab' and USE_DRIVE:
    from pathlib import Path
    from google.colab import drive
    drive.mount('/content/drive')
    store = Path('/content/drive/MyDrive/methane_dimer_36_data')
    store.mkdir(parents=True, exist_ok=True)
    local = Path('data')
    if local.exists() or local.is_symlink():
        assert local.is_symlink(), 'data/ already exists as a real directory; move it first'
    else:
        local.symlink_to(store, target_is_directory=True)
    print('data ->', local.resolve())
elif HOST == 'colab':
    print('NOT persisting; data/ dies with this session. Set USE_DRIVE = True first.')
else:
    print(f'{HOST}: working under {BASE}, which persists. Nothing to do.')

## 3 - Self-test and cost model

Neither needs the chemistry stack. Run them first: they catch version drift
before any CPU time is spent. Seconds.

`{sys.executable}` rather than a bare `python` so the subprocess is the same
interpreter as this kernel - on qBraid those differ.

In [ ]:
!{sys.executable} run.py selftest --quiet
!{sys.executable} run.py plan

## 4 - END-TO-END VALIDATION (run this before anything expensive)

Runs the **entire chemistry pipeline** - RHF, AVAS, cache round-trip, CCSD, LUCJ,
ffsim sampling, SQD, variance, ablation, binding energy - on a tiny STO-3G active
space. Same molecule, same functions, seconds to run.

The load-bearing check: when the subspace saturates the CAS, **SQD must equal
CASCI to 1e-8 Ha** - solver precision, not chemical accuracy. If the SQD and
CASCI Hamiltonians ever differ, this catches it immediately.

**If this fails, stop and fix it. Nothing below will be right.**

In [ ]:
!{sys.executable} run.py validate

## 5 - Geometry

The paper publishes no coordinates and never names the monomer orientation.
D3d is the default; change it here and every downstream cache key changes with it.

In [ ]:
ORIENTATION = 'd3d'
DISTANCE    = 3.638   # the paper's extra point, and its equilibrium

import geometry, paper
atoms = geometry.methane_dimer(DISTANCE, orientation_name=ORIENTATION)
geometry.check_geometry(atoms, expected_distance=DISTANCE)
print(geometry.to_xyz(atoms, f'methane dimer {ORIENTATION} R={DISTANCE}'))

## 6 - Hamiltonian: RHF/aug-cc-pVQZ + AVAS

528 basis functions, so density fitting is on. AVAS on `C[2s,2p], H[1s]` must
land on exactly (16e,16o); `chemistry.py` asserts it rather than tuning to it.

**First run: tens of minutes**, and longer on 2 vCPUs. Re-runs hit `data/cache`
and return immediately.

In [ ]:
from pathlib import Path
import chemistry, spaces

CACHE = Path('data/cache')
spec  = chemistry.SystemSpec(distance=DISTANCE, orientation=ORIENTATION)
mol_data = chemistry.load_or_build(spec, CACHE, verbose=4)

print(f'HF           {mol_data.hf_energy:.10f} Ha')
print(f'active       ({sum(mol_data.nelec)}e,{mol_data.norb}o)')
print(f'determinants {spaces.n_determinants(mol_data.norb, mol_data.nelec):,}')

## 7 - CCSD amplitudes and the LUCJ ansatz

Cheap: `MolecularData.scf` round-trips the *active-space* integrals through an
FCIDUMP, so this is a 16-orbital CCSD, not a 528-orbital one.

In [ ]:
import ansatz, reference

e_ccsd = reference.run_ccsd(mol_data, store_amplitudes=True)
print(f'CCSD (active space) {e_ccsd:.10f} Ha')

layout = ansatz.heavy_hex_layout(mol_data.norb, n_reps=2)
ansatz.validate_layout(layout)      # asserts 32 + 4 = 36
print(layout.to_dict())

operator = ansatz.build_operator(mol_data, layout)
circuit  = ansatz.build_circuit(mol_data, operator)
print(f'circuit: {circuit.num_qubits} qubits, depth {circuit.depth()}')

## 8 - Sample

`FfsimSampler` implements the SamplerV2 interface but simulates inside the (8,8)
sector: 2.47 GiB rather than the 64 GiB a dense 32-qubit statevector would need.

Swap in `sampling.HardwareSampler(backend)` for a real device. Nothing downstream
changes.

In [ ]:
import sampling

SHOTS = paper.TOTAL_SAMPLES      # 200,000
sampler = sampling.NoiselessSampler(mol_data.norb, mol_data.nelec, seed=12345)
sampled = sampler.sample(circuit, SHOTS)

valid    = sampling.valid_configuration_fraction(sampled.bit_array, mol_data.norb, mol_data.nelec)
baseline = sampling.random_validity_probability(mol_data.norb, mol_data.nelec)
print(f'valid {valid:.2%}   random baseline {baseline:.2%}')
print()
print('NOTE: at half filling the random baseline is percent-level, so validity')
print('      fraction is a weak diagnostic here. Section 11 is the real control.')

## 9 - SQD

Start on the `extrapolation-low` rung (|chi_b| = 9e3). The `converged` rung
reproduces Table II verbatim but needs a large-memory machine - see `run.py plan`.

The addon's default solver runs the batches **sequentially**, so peak memory is
one batch rather than ten - but it also means 10 batches x 10 recovery steps =
100 diagonalizations back to back. Budget an hour or more, several on 2 vCPUs,
and make sure section 2b is sorted before you start.

In [ ]:
import sqd

rung   = spaces.rung('extrapolation-low')
config = sqd.SqdConfig(samples_per_batch=rung.samples_per_batch,
                       n_batches=rung.n_batches,
                       max_iterations=paper.RECOVERY_STEPS,
                       max_dim=rung.max_dim, seed=12345)

result = sqd.run(mol_data, sampled.bit_array, config, source='noiseless')
print()
print(f'SQD  E = {result.energy:.10f} Ha')
print(f'     d = {result.subspace_dimension:,} ({result.subspace_fraction:.2%} of CAS)')

## 10 - CASCI reference (large memory)

165,636,900 determinants. One CI vector is 961 MiB; budget ~15 GiB. The guard
below refuses to start rather than letting the kernel get OOM-killed an hour in.

In [ ]:
import os
import binding

ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
if ram_gib < 15:
    raise MemoryError(
        f'CASCI(16e,16o) needs ~15 GiB and this machine has {ram_gib:.1f} GiB. '
        'Colab: Runtime -> Change runtime type -> High-RAM. '
        'qBraid: Pro tier, Large session machine (8 vCPU / 25 GB). '
        'Then re-run from section 2; the cache survives if section 2b applies.')

e_casci = reference.run_casci(mol_data, verbose=4)
print(f'CASCI {e_casci:.10f} Ha')

binding.variational_check(result.energy, e_casci)   # SQD may not fall below CASCI
agreement = binding.Agreement(result.energy, e_casci)
print(agreement)
print(f'paper target at |chi_b|=20e3: {paper.SQD_VS_CASCI_TARGET_KCAL} kcal/mol')

## 11 - Ablation: the measurement that matters

Uniform random configurations at **matched subspace dimension**. The energy gap
is the quantum layer's contribution, as a number. A control at a different
dimension compares two things at once and settles nothing.

This one does not need CASCI, so it is reachable on every tier that got you
through section 9.

In [ ]:
control_cfg = sqd.ablation_config(result, config)
uniform     = sampling.UniformSampler(mol_data.norb, mol_data.nelec, seed=12345)
control     = sqd.run(mol_data, uniform.sample(shots=SHOTS).bit_array,
                      control_cfg, source='uniform')

gap = (control.energy - result.energy) / binding.MILLIHARTREE
print(f'SQD      {result.energy:.10f} Ha   d = {result.subspace_dimension:,}')
print(f'uniform  {control.energy:.10f} Ha   d = {control.subspace_dimension:,}')
print(f'gap      {gap:+.4f} mHa')

## 12 - Scan the PES, verify, report

Binding energy is `E(R) - E(48 A)` (paper Eq. 2), so the 48 A point is required,
not optional.

**Do not run `--all` here.** It is 16 distances, each an aug-cc-pVQZ RHF plus an
SQD run, and `reference --all` is 16 CASCI diagonalizations at ~15 GiB apiece:
days of compute. The cell below does the two points `verify` actually needs - the
equilibrium and the 48 A separation - and that is already several hours. Make
sure section 2b is sorted first, then widen the grid a point at a time across
sessions.

In [ ]:
for R in (3.638, 48.000):
    print()
    print('=' * 60)
    print(f'R = {R} A')
    print('=' * 60)
    !{sys.executable} run.py sqd       --distance {R} --rung extrapolation-low
    !{sys.executable} run.py reference --distance {R}

In [ ]:
!{sys.executable} run.py verify
!{sys.executable} run.py report

The full grid, for a machine that can take it:

```bash
python run.py sqd       --all --rung extrapolation-low
python run.py reference --all
python run.py verify
python run.py report
```